# Práctica 1.1 — Probabilidad y Estadística
## Análisis de la evolución poblacional mundial

### Objetivo
Utilizar un conjunto de datos real para comprobar el funcionamiento del entorno de análisis de datos y aplicar medidas estadísticas descriptivas básicas con Pandas y NumPy.

## Actividad 1 — Identificación de la fuente

Para esta actividad se consultaron los archivos `data/raw/annual-population-growth/annual-population-growth.readme.md` y `annual-population-growth.metadata.json`. Se trabajará únicamente con los datos históricos de cambio anual de población (1951–2023).

### Fuente y cita

Los datos utilizados provienen de **Our World in Data (OWID)**. La fuente original es **United Nations, World Population Prospects (2024)**.

> UN, World Population Prospects (2024) – processed by Our World in Data. “Annual change in population – UN WPP” [dataset]. United Nations, “World Population Prospects”; United Nations, “World Population Prospects - Interim Update” [original data].

### Respuestas

**¿Quién produjo originalmente los datos?**
Las Naciones Unidas, mediante *World Population Prospects (2024)*.

**¿Qué organización los procesa para su utilización en Our World in Data?**
Our World in Data (OWID).

**¿Qué representa la variable que analizarás?**
La variable representa el cambio anual neto de población, calculado como la diferencia entre la población al 1 de julio de dos años consecutivos. Incluye conjuntamente el efecto de nacimientos, defunciones y migración; su unidad es personas.

## Actividad 2 — Carga y exploración de datos

En esta actividad se carga el archivo de cambio anual de población y se revisa la estructura del DataFrame. Aunque el conjunto incluye proyecciones, en las actividades posteriores se utilizarán únicamente los registros históricos.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
path = Path("../data/raw/annual-population-growth/annual-population-growth.csv")

df = pd.read_csv(path)
df.head()

,Entity,Code,Year,Annual change in population,Annual population change (Projected)
0,Afghanistan,AFG,1951,103163.0,NaN
1,Afghanistan,AFG,1952,108441.0,NaN
2,Afghanistan,AFG,1953,108919.0,NaN
3,Afghanistan,AFG,1954,111251.0,NaN
4,Afghanistan,AFG,1955,119027.0,NaN


### Exploración de la estructura

Las siguientes celdas muestran las primeras filas, dimensiones, columnas, tipos de datos y estadísticas descriptivas solicitadas.

In [3]:
print("Dimensiones (filas, columnas):", df.shape)
print("\nNombres de columnas:")
print(df.columns.tolist())
print("\nTipos de datos:")
print(df.dtypes)

df.describe()

Dimensiones (filas, columnas): (38400, 5)

Nombres de columnas:
['Entity', 'Code', 'Year', 'Annual change in population', 'Annual population change (Projected)']

Tipos de datos:
Entity                                      str
Code                                        str
Year                                      int64
Annual change in population             float64
Annual population change (Projected)    float64
dtype: object


,Year,Annual change in population,Annual population change (Projected)
count,38400.000000,1.868800e+04,1.971200e+04
mean,2025.500000,2.063019e+06,8.375481e+05
std,43.300872,9.440859e+06,5.816745e+06
min,1951.000000,-3.315935e+06,-2.291035e+07
25%,1988.000000,1.781000e+03,-9.742750e+03
50%,2025.500000,4.689800e+04,9.350000e+01
75%,2063.000000,3.021128e+05,1.411160e+05
max,2100.000000,9.337137e+07,7.206736e+07


In [ ]:
print("Valores faltantes por columna:")
print(df.isnull().sum())
print("\nNúmero de entidades diferentes:", df["Entity"].nunique())

### Respuestas

**¿Cuántas filas y columnas tiene el dataset?**  \n
El dataset tiene **38,400 filas** y **5 columnas**.

**¿Qué tipo de datos contiene la variable `Year`?**  \n
Al cargar el archivo con Pandas, `Year` tiene tipo entero (`int64`).

**¿Existen valores faltantes?**  \n
Sí. `Code` tiene 1,200 valores faltantes. Las dos columnas de cambio poblacional también presentan valores faltantes porque una contiene los datos históricos y la otra las proyecciones; se utilizará la columna histórica en la siguiente actividad.

**¿Cuántas entidades diferentes contiene el dataset?**  \n
El dataset contiene **256 entidades** diferentes.

In [6]:
## Extracción de entidad y cambio anual de población

entity_annual_change = (
    df[["Entity", "Annual change in population"]]
    .dropna(subset=["Annual change in population"])
    .reset_index(drop=True)
)

entity_annual_change.head()

,Entity,Annual change in population
0,Afghanistan,103163.0
1,Afghanistan,108441.0
2,Afghanistan,108919.0
3,Afghanistan,111251.0
4,Afghanistan,119027.0


In [7]:
df["Entity"].unique()

<StringArray>
[        'Afghanistan',         'Africa (UN)',             'Albania',
             'Algeria',      'American Samoa',       'Americas (UN)',
             'Andorra',              'Angola',            'Anguilla',
 'Antigua and Barbuda',
 ...
             'Vanuatu',             'Vatican',           'Venezuela',
             'Vietnam',   'Wallis and Futuna',      'Western Sahara',
               'World',               'Yemen',              'Zambia',
            'Zimbabwe']
Length: 256, dtype: str

In [14]:
belgium_1983_2021 = (
    df.loc[
        (df["Entity"] == "Belgium") & df["Year"].between(1983, 2021),
        ["Entity", "Year", "Annual change in population"]
    ]
    .sort_values("Year")
)

print("Belgium: annual population change (1983–2021)")
display(belgium_1983_2021)

annual_change = "Annual change in population"
min_row = belgium_1983_2021.loc[belgium_1983_2021[annual_change].idxmin()]
max_row = belgium_1983_2021.loc[belgium_1983_2021[annual_change].idxmax()]

summary = pd.DataFrame({
    "Statistic": ["Minimum", "Maximum"],
    "Year": [int(min_row["Year"]), int(max_row["Year"])],
    annual_change: [min_row[annual_change], max_row[annual_change]]
})

print("Minimum and maximum annual change, including the year:")
display(summary)

print("Standard statistics for the 1983–2021 range:")
display(belgium_1983_2021[annual_change].describe().to_frame(annual_change))

Belgium: annual population change (1983–2021)


,Entity,Year,Annual change in population
3332,Belgium,1983,6807.0
3333,Belgium,1984,5450.0
3334,Belgium,1985,5752.0
3335,Belgium,1986,6643.0
3336,Belgium,1987,12169.0
3337,Belgium,1988,17416.0
3338,Belgium,1989,20009.0
3339,Belgium,1990,24393.0
3340,Belgium,1991,37994.0
3341,Belgium,1992,47026.0


Minimum and maximum annual change, including the year:


,Statistic,Year,Annual change in population
0,Minimum,1984,5450.0
1,Maximum,2010,123683.0


Standard statistics for the 1983–2021 range:


,Annual change in population
count,39.000000
mean,43808.051282
std,28197.423435
min,5450.000000
25%,22608.000000
50%,43287.000000
75%,57480.000000
max,123683.000000
